## A custom integration term with `LocalOperator`

Builds on [06_multipatch_stiffness](06_multipatch_stiffness.ipynb). `PatchIntegrator` ships two built-in kernels -- stiffness and consistent mass -- each implemented in C++. `LocalOperator` is a third way in: subclass it **from Python** and override `compute_integrand(R, dRdx, dRdy)`.

`PatchIntegrator` still does the Gauss-point loop and the Jacobian/gradient computation -- the same machinery it uses internally for the built-in kernels. The operator only has to return the *term* to integrate at one Gauss point (e.g. `B^T*D*B` for stiffness, `rho*N^T*N` for mass), built from the basis values `R` and physical gradients `dRdx`, `dRdy` it is handed. `PatchIntegrator.integrate_operator()`/`assemble_operator()` then multiply that term by the Gauss weight and `|detJ|` and sum it over every Gauss point of every span.

This is meant for development/testing convenience -- prototyping a new physical term without touching C++ or rebuilding the extension -- not for performance (every Gauss point triggers a Python call). For performance-critical assembly, prefer `integrate_stiffness()`/`integrate_mass()`.

This notebook does two things: reproduce the existing stiffness term from Python (to check the mechanism works and to show what's available), then write a genuinely new term -- a lumped mass matrix -- that has no C++ counterpart.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from yeti_iga.future.bspline import (BSpline, BSplineSurface, ControlPointManager,
    Patch, GlobalDOFManager, PatchDOFManager, PatchAssembly, IGABasis1D,
    PatchIntegrator, MaterialProperties, LocalOperator)
from yeti_iga.future.plotting import plot_patches_2d

### Reproducing the stiffness term from Python

`compute_integrand(R, dRdx, dRdy)` receives the basis values and physical-space gradients of the span's active basis functions at one Gauss point (`u`-fastest order, matching the row order of `patch.control_points_for_span()`). Building `B` and `D` and returning `B^T*D*B` is all that's needed -- no Gauss loop, no Jacobian, no weighting.

The operator carries its own parameters (here `E`, `nu`) -- it has no dependency on `MaterialProperties`, unlike the built-in kernels.

In [ ]:
class PythonStiffnessOperator(LocalOperator):
    """Pure-Python re-implementation of the built-in B^T*D*B stiffness term."""

    def __init__(self, E, nu):
        super().__init__()
        self.E = E
        self.nu = nu

    def compute_integrand(self, R, dRdx, dRdy):
        nb_loc = len(R)
        factor = self.E / (1.0 - self.nu ** 2)
        D = np.array([
            [factor, factor * self.nu, 0.0],
            [factor * self.nu, factor, 0.0],
            [0.0, 0.0, factor * (1.0 - self.nu) / 2.0],
        ])

        B = np.zeros((3, 2 * nb_loc))
        B[0, 0::2] = dRdx
        B[1, 1::2] = dRdy
        B[2, 0::2] = dRdy
        B[2, 1::2] = dRdx

        return B.T @ D @ B

### A single patch, the same rectangle as before

Same 6x1, degree-2, two-element geometry as [06_multipatch_stiffness](06_multipatch_stiffness.ipynb), already validated against the legacy Fortran solver.

In [ ]:
mgr = ControlPointManager(dim=2)
for y in (0.0, 0.5, 1.0):
    for x in (0.0, 1.5, 3.0, 4.5, 6.0):
        mgr.add_point([x, y])

su = BSpline(2, np.array([0., 0., 0., 0.5, 0.5, 1., 1., 1.]))
sv = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
mapping = list(range(15))

dof_manager = GlobalDOFManager([2] * mgr.n_points)
patch_dof_manager = PatchDOFManager(2, mapping, dof_manager)
patch = Patch(BSplineSurface(su, sv), mgr, mapping, [5, 3], patch_dof_manager)

basis_u = IGABasis1D.build(su, 3)
basis_v = IGABasis1D.build(sv, 3)
material = MaterialProperties(210000, 0.3)

integrator = PatchIntegrator(patch, basis_u, basis_v, material)
K_builtin = integrator.integrate_stiffness()
K_operator = integrator.integrate_operator(PythonStiffnessOperator(210000, 0.3))

diff = np.abs(K_builtin.toarray() - K_operator.toarray())
print('max abs difference vs. integrate_stiffness():', diff.max())
assert np.allclose(K_builtin.toarray(), K_operator.toarray(), rtol=1.e-8, atol=1.e-8)
print('integrate_operator() == integrate_stiffness(): OK')

### A genuinely new term: lumped mass

Only the *consistent* mass matrix is implemented in C++ (`integrate_mass()`). A row-sum ("HRZ") lumped mass has no built-in counterpart -- this is the kind of term `LocalOperator` is for: try it out without touching C++.

Row-summing a consistent mass term `outer(R, R)` at a single Gauss point gives `R` itself -- the span's active basis functions already sum to 1 there (partition of unity) -- so lumping reduces to a per-Gauss-point diagonal term, no need to wait for the full span integral before lumping.

Checked two ways: the result must be diagonal, and the total mass must be conserved (equal to `rho * area`).

In [ ]:
class LumpedMassOperator(LocalOperator):
    """Row-sum lumped mass -- not one of PatchIntegrator's built-in kernels."""

    def __init__(self, rho):
        super().__init__()
        self.rho = rho

    def compute_integrand(self, R, dRdx, dRdy):
        nb_loc = len(R)
        diag = self.rho * np.asarray(R)
        M_loc = np.zeros((2 * nb_loc, 2 * nb_loc))
        M_loc[0::2, 0::2] = np.diag(diag)
        M_loc[1::2, 1::2] = np.diag(diag)
        return M_loc


rho = 7800.0
M_lumped = integrator.integrate_operator(LumpedMassOperator(rho)).toarray()

off_diag = M_lumped - np.diag(np.diag(M_lumped))
print('max abs off-diagonal term:', np.abs(off_diag).max())

area = 6.0 * 1.0
total_mass = np.trace(M_lumped[0::2, 0::2])
print(f'total mass: {total_mass:.3f}  (rho * area = {rho * area:.3f})')
assert np.allclose(off_diag, 0.0, atol=1.e-10)
assert np.isclose(total_mass, rho * area, rtol=1.e-6)
print('lumped mass is diagonal and conserves total mass: OK')

In [ ]:
plt.figure(figsize=(4, 4))
plt.spy(M_lumped, markersize=4)
plt.title('Sparsity pattern of the lumped mass matrix')
plt.show()

### Multipatch: `assemble_operator()`

Same as `assemble_stiffness()`/`assemble_mass()`, but for a custom `LocalOperator` -- one operator instance per patch, in `add_patch()` order. Splitting the rectangle into two patches sharing the middle edge (same construction as [06_multipatch_stiffness](06_multipatch_stiffness.ipynb)) must not change the assembled physics.

In [ ]:
mgr_multi = ControlPointManager(dim=2)
for y in (0.0, 0.5, 1.0):
    for x in (0.0, 1.5, 3.0, 4.5, 6.0):
        mgr_multi.add_point([x, y])

dofs_per_control_point = [2] * mgr_multi.n_points
global_dof_manager = GlobalDOFManager(dofs_per_control_point)

su_left = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
sv_left = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
mapping_left = [0, 1, 2, 5, 6, 7, 10, 11, 12]
dof_manager_left = PatchDOFManager(2, mapping_left, global_dof_manager)
patch_left = Patch(BSplineSurface(su_left, sv_left), mgr_multi, mapping_left, [3, 3],
                    dof_manager_left)

su_right = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
sv_right = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
mapping_right = [2, 3, 4, 7, 8, 9, 12, 13, 14]
dof_manager_right = PatchDOFManager(2, mapping_right, global_dof_manager)
patch_right = Patch(BSplineSurface(su_right, sv_right), mgr_multi, mapping_right, [3, 3],
                     dof_manager_right)

assembly = PatchAssembly()
assembly.add_patch(patch_left)
assembly.add_patch(patch_right)
assembly.detect_shared_control_points()

plot_patches_2d(assembly, show_control_points=True, show_control_point_indices=True,
                title='Two patches sharing the x=3 edge')

In [ ]:
operators = [PythonStiffnessOperator(210000, 0.3), PythonStiffnessOperator(210000, 0.3)]
K_multi_operator = PatchIntegrator.assemble_operator(assembly, operators)

diff = np.abs(K_builtin.toarray() - K_multi_operator.toarray())
print('max abs difference vs. the single-patch reference:', diff.max())
assert np.allclose(K_builtin.toarray(), K_multi_operator.toarray(), rtol=1.e-8, atol=1.e-8)
print('assemble_operator() == single-patch integrate_stiffness(): OK')